In [2]:
from unsloth import FastVisionModel
from dotenv import load_dotenv
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from jiwer import wer, cer
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

In [3]:
load_dotenv()

True

In [4]:
# TODO: update with front/back details
field_structure = {
    "first_name": "string (arabic)",
    "last_name": "string (arabic)",
    "national_id": "string, 14 digits",
    "address": "string (arabic)",
    "birthdate": "string, formatted date (arabic)",
    "issue_date": "string, formatted date (arabic)",
    "expiration_date": "string, formatted date (arabic)",
    "job_title": "string (arabic)",
    "gender": "string, 'male' or 'female' (arabic)",
    "religion": "string, 'muslim' or 'christian' (arabic)",
    "marital_status": "string, 'single', 'married' or 'widow' (arabic)",
    "governorate": "string (arabic)"
}

In [5]:
SYSTEM_PROMPT = f'''
    You are a Vision Language Model tasked with extracted field values from an Egyptian national identity
    document. You must extract the fields without making any changes to the fields and return them
    as they are in Arabic script. If there is something you cannot extract, do not attempt to infer it based
    on other information.
'''
USER_PROMPT = f'''
    You are given two sides of an Egyptian National ID, front and back.
    Extract all the fields, regardless of the side they are found on, out of the ID 
    following this format: {field_structure}.  The key order does not matter. Return all 
    fields as they appear and do not make any changes or updates to any of the fields. Return in json format.
'''

In [6]:
def generate_conversation(data):
    image_front = Image.open(f"./data/synthetic-ids/images/ID{data['image']}.png")
    image_back = Image.open(f"./data/synthetic-ids/images/IDB{data['image']}.png")
    
    first_name = data['first_name']
    last_name = data['last_name']
    gender = data['gender']
    national_id = data['national_id']
    address = data['address']
    issue_date = data['issue_date']
    expiration_date = data['expiration_date']
    job_title = data['job_title']
    birthdate = data['birthdate']
    religion = data['religion']
    marital_status = data['marital_status']
    governorate = data['governorate']

    message = f'''{{"first_name": {first_name},
        "last_name": {last_name},
        "national_id": {national_id},
        "address": {address},
        "birthdate": {birthdate},
        "issue_date": {issue_date},
        "expiration_date": {expiration_date},
        "gender": {gender},
        "job_title": {job_title},
        "religion": {religion},
        "marital_status": {marital_status},
        "governorate": {governorate}}}'''

    conversation = [
        {
            'role': 'system',
            'content': [
                    {
                        'type': 'text',
                        'text': SYSTEM_PROMPT
                    }
            ]
        },
        {
            'role': 'user', 
            'content': [
                {
                    'type': 'text', 'text': USER_PROMPT 
                },
                {
                    'type': 'image', 
                    'image': image_front
                },
                {
                    'type': 'image',
                    'image': image_back
                }
            ]
        },
        {
            'role': 'assistant', 
            'content': [
                {
                    'type': 'text', 
                    'text': message
                }
            ]
        }
    ]
    return conversation

##### Load and split data

In [7]:
dataset = pd.read_csv("./data/synthetic-ids/IDLabels.csv")

In [8]:
# TODO: update to load images and fields separately
train, val = train_test_split(dataset, test_size=0.25)

#### Apply chat transformation

In [9]:
training_data = []
for idx, sample in train.iterrows():
    training_data.append(generate_conversation(sample))

In [10]:
validation_data = []
for idx, sample in val.iterrows():
    validation_data.append(generate_conversation(sample))

#### Define inference functions

In [11]:
def infer(model, tokenizer, sample):
    FastVisionModel.for_inference(model)
    
    messages = [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'image'
                }, 
                {
                    'type': 'text',
                    'text': USER_PROMPT
                }
            ]
        }
    ]

    input_text = tokenizer.apply_chat_templates(messages, add_generation_prompt=True)
    inputs = tokenizer(sample, input_text, add_special_tokens=False, return_tensors='pt').to(model.device)

    inference = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=1.5, min_p=0.1)

    return inference

In [12]:
def batch_infer(model, tokenizer, samples):
    FastVisionModel.for_inference(model)
    predictions = []
    
    for sample in tqdm(samples):
        prediction = infer(model, tokenizer, sample)
        predictions.append(prediction)
        
    return predictions


#### Define evaluation metrics

In [13]:
def eval(model, tokenizer, samples):
    sample_images = [Image.open(f"./images/{s['image']}.png") for s in samples]
    predictions = batch_infer(model, tokenizer, sample_images)

    field_correct = {key: 0 for key in field_structure}
    field_avg_wer = {key: 0 for key in field_structure if key != "national_id" and "date" not in key} # no words in national_id or any date field
    field_avg_cer = {key: 0 for key in field_structure}
    total = len(samples)

    assert len(samples) == len(predictions)

    for sample, prediction in tqdm(zip(samples, predictions)):
        for key in field_correct:
            field_correct[key] += 1 if prediction[key] == sample[key] else 0

        for key in field_avg_wer:
            field_avg_wer[key] += wer(prediction[key], sample[key])

        for key in field_avg_cer:
            field_avg_cer[key] += cer(prediction[key], sample[key])
        

    for key in field_correct:
        field_correct[key] /= total
        field_avg_cer[key] /= total
    for key in field_avg_wer:
        field_avg_cer[key] /= total

    return {"correct_match": field_correct, "avg_cer": field_avg_cer, "avg_wer": field_avg_wer}
        

##### Load pretrained model

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-8B-Instruct",
                                                   load_in_4bit = True,
                                                   use_gradient_checkpointing=True)


# model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3.5-0.8B",
#                                                    load_in_4bit = True,
#                                                    use_gradient_checkpointing=True)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Error in sys.excepthook:
Error in sys.excepthook:
Traceback (most recent call last):
  File "/Users/zain/ocr-id-parsing/.venv/lib/python3.14/site-packages/ipykernel/iostream.py", line 217, in _event_pipe
    event_pipe = self._local.event_pipe
                 ^^^^^^^^^^^^^^^^^^^^^^
AttributeError: '_thread._local' object has no attribute 'event_pipe'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/zain/ocr-id-parsing/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 2112, in excepthook
    self.showtraceback((etype, value, tb), tb_offset=0)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zain/ocr-id-parsing/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 2234, in showtraceback
    self._showtraceback(etype, value, stb)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/Users/zain/ocr-id-parsing/.venv/lib/python3.14/site-packages/ipykernel/

##### Pretrained model metrics (baseline)

In [ ]:
eval(model, tokenizer, val)

##### Set up finetuning model

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)
FastVisionModel.for_training(model)

In [ ]:
args = SFTConfig(
        # training
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        learning_rate = 2e-4, 
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",

        # eval
        per_device_eval_batch_size = 1,
        eval_strategy='steps',
        eval_steps=50,

        # output
        output_dir = "models",
        report_to = 'wandb',
        run_name='ocr-id-detection',

        # logging
        logging_steps = 25,
        save_steps=50,

        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 10000,
        bf16=True,

        push_to_hub=True,
        hub_private_repo=True,
        hub_model_id='zain110506/ocr-id-parser',
        hub_strategy='checkpoint'
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_data,
    eval_dataset=validation_data,
    args=args
)


#### Train the model

In [ ]:
trainer_stats = trainer.train()

#### Save the model and tokenizer

In [ ]:
model.save_pretrained("qwen3_vlm")
tokenizer.save_pretrained("qwen3_vlm")